In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
from thesis.common.config import TYPE_TEST, TYPE_TRAIN
from thesis.common.logger import setup_logger
from thesis.eta.config import SCENARIOS_SPECS
from thesis.eta.data import generate_trips, load_fcd_dataset
from thesis.eta.experiment import initialize_experiment, save_model, save_results
from thesis.eta.features import split_features_and_target
from thesis.eta.models import get_baseline_models
from thesis.eta.pipeline import evaluate_predictions, make_predictions, train_model

In [20]:
experiment_name = "playground"
artifacts_dir, logs_dir, plots_dir, results_dir = initialize_experiment(experiment_name)
logger = setup_logger(experiment_name, logs_dir)

scenario_name, train_path, test_path = SCENARIOS_SPECS[0]
logger.info(f"Starting scenario {scenario_name}")

train_dataset_id = f"{scenario_name}-{TYPE_TRAIN}"
test_dataset_id = f"{scenario_name}-{TYPE_TEST}"

fcd_train = load_fcd_dataset(train_path)
# fcd_test = load_fcd_dataset(test_path)
# fcd_train = preprocess_fcd_dataset(fcd_train)
# fcd_test = preprocess_fcd_dataset(fcd_test)
# trips_train = generate_trips(fcd_train)
# trips_test = generate_trips(fcd_test)

2025-07-09 21:51:45 - playground - INFO - Starting experiment playground
2025-07-09 21:51:45 - playground - INFO - Starting scenario base
2025-07-09 21:51:45 - thesis.eta.data - INFO - File base-train-fcd.csv not found, downloading from Zenodo


base-train-fcd.csv: 100%|██████████| 709M/709M [01:04<00:00, 11.0MB/s]  


2025-07-09 21:52:51 - thesis.eta.data - INFO - File base-train-fcd.csv is valid
2025-07-09 21:52:56 - thesis.eta.data - INFO - Loaded 12399995 rows of FCD data from C:\Users\george\Desktop\diploma-thesis\simulation\data\base-train-fcd.csv


In [4]:
trips_train_augmented = generate_trips(fcd_train, augmentation_rate=0.01)
trips_train_augmented

2025-07-02 13:30:56 - thesis.eta.data - INFO - Generated 54285 full trips
2025-07-02 13:31:25 - thesis.eta.data - INFO - Generated 93170 sub-trips
2025-07-02 13:31:25 - thesis.eta.data - INFO - Generated 147455 trips


,source_x,source_y,destination_x,destination_y,hour_bin,distance,duration
0,2141.68,730.24,1655.40,286.54,0,1020.28,96
1,978.03,262.45,322.22,1481.38,0,1797.01,249
2,2264.56,411.90,1041.00,922.65,0,2174.41,245
3,814.76,1345.18,328.07,521.07,0,1778.26,235
4,181.77,1408.68,841.76,1223.66,0,739.86,92
...,...,...,...,...,...,...,...
147450,2360.54,1131.29,1378.06,124.43,1,1847.09,203
147451,2056.20,599.41,1367.93,125.76,1,1144.43,136
147452,1360.02,285.82,1710.04,1378.93,1,2072.98,328
147453,1069.07,585.14,1710.04,1378.93,1,1267.54,221


In [8]:
trips_train = trips_train_augmented

In [9]:
X_train, y_train = split_features_and_target(trips_train)
X_test, y_test = split_features_and_target(trips_test)

models = get_baseline_models()
results = {}
for model_name, model in models.items():
    training_results = train_model(model, model_name, X_train, y_train)
    predictions, prediction_results = make_predictions(model, model_name, X_test)
    evaluation_results = evaluate_predictions(y_test, predictions, model_name)

    results[model_name] = {**training_results, **prediction_results, **evaluation_results}
    save_model(model, model_name, scenario_name, artifacts_dir)

save_results(results, scenario_name, results_dir)

2025-07-02 02:10:23 - thesis.eta.features - INFO - Split 147455 samples into 6 features and 1 target variable
2025-07-02 02:10:23 - thesis.eta.features - INFO - Split 54183 samples into 6 features and 1 target variable
2025-07-02 02:10:23 - thesis.eta.pipeline - INFO - lr - Training: 0.009s
2025-07-02 02:10:23 - thesis.eta.pipeline - INFO - lr - Prediction: 0.002s
2025-07-02 02:10:23 - thesis.eta.pipeline - INFO - lr - Evaluation: 0.003s, MAE: 40.11s, MSE: 6418.31s, RMSE: 80.11s, MAPE: 18.48%, R2: 0.544
2025-07-02 02:10:23 - thesis.eta.experiment - INFO - Model saved to C:\Users\george\Desktop\diploma-thesis\outputs\playground\artifacts\base\lr.joblib
2025-07-02 02:10:23 - thesis.eta.pipeline - INFO - xgboost - Training: 0.575s
2025-07-02 02:10:23 - thesis.eta.pipeline - INFO - xgboost - Prediction: 0.009s
2025-07-02 02:10:23 - thesis.eta.pipeline - INFO - xgboost - Evaluation: 0.005s, MAE: 36.63s, MSE: 6229.39s, RMSE: 78.93s, MAPE: 16.22%, R2: 0.558
2025-07-02 02:10:23 - thesis.eta.ex